In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import seaborn as sns
import re
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")

In [ ]:
url = "https://groww.in/stocks/filter"
# its the groww's stock filter page.
# storing the URL of the webpage we want to access.

In [ ]:
response = requests.get(url)
response
# use the requests library to make an HTTP GET request to the URL.
# the result is stored in the "response" object, which contains data and metadata about the request.


In [ ]:
response.status_code
# shows the HTTP status code of the response.
# 200 means the request was successful, 404 means page not found, 403 means forbidden, etc.
response.text
# Contains the content of the response as a string (HTML or JSON text).
# If it’s a webpage, this will be the HTML source code.
# response.json()

# Since https://groww.in/stocks/filter is a website (not a JSON API),
# response.text will give you the HTML, 
# and response.json() would likely throw an error. 
# For web scraping, you’ll usually parse response.text with BeautifulSoup.

In [ ]:
from bs4 import BeautifulSoup # parses HTML content so we can extract data from a webpage
import requests #  Library to make HTTP requests (GET/POST)
import numpy as np
import time # used to measure execution time of each page scraping loop

companylist=[]    # name of the company
marketpricelist=[]  # stores current market price
closepricelist=[]   # stores last days closing price
pagenum=[] # tracks which page the data came from
marketcaplist = []  # market cap of the company
dailylist = [] #  Stores positive daily price changes
daily_neglist = [] #  Stores negative daily price changes

total_time = time.time() # Record start time for the entire scraping process
# time.time() returns the current time in seconds.
for i in range(0,333): # we have totally 0 to 330 (in url pageno)
    start_time = time.time() # start timer for current page
    # Construct the URL dynamically based on page number and filters
    URL = f"https://groww.in/stocks/filter?closePriceHigh=500000&closePriceLow=0&marketCapHigh=3000000&marketCapLow=0&page={i}"
    page = requests.get(URL) # Send GET request to the URL and get response
    soup = BeautifulSoup(page.text, "html.parser") # parse HTML content for the page.

    # selects all table rows in the stock table
    rows = soup.select("table.tb10Table.borderPrimary tbody tr")

    # loop through each row in the table
    for row in rows:
        # Extract company name from <span> tag with class 'st76SymbolName'
        company = row.find('span', attrs={'class':'st76SymbolName'})

        # Extract market price from <div> tag with class 'st76CurrVal bodyBaseHeavy'
        marketprice = row.find('div', attrs={'class':'st76CurrVal bodyBaseHeavy'}) 

        # to get the close price and market cap they are under the same class name, so 
        # we used find_all and used indexing to get the values
        tds = row.find_all('td', class_='contentPrimary st76Pad16 bodyBase')

        closeprice = tds[0] 
        marketcap = tds[1] 

        # Extract daily positive and negative price changes
        daily = row.find('div', attrs={'class':'bodySmallHeavy contentPositive'})
        daily_neg = row.find('div', attrs={'class':'bodySmallHeavy contentNegative'})

        # Append extracted values to corresponding lists, using np.nan if missing
        companylist.append(company.text if company else np.nan)
        marketpricelist.append(marketprice.text if marketprice else np.nan)
        closepricelist.append(closeprice.text.strip() if closeprice else np.nan)
        marketcaplist.append(marketcap.text.strip() if marketcap else np.nan)
        dailylist.append(daily.text.strip() if daily else np.nan)
        daily_neglist.append(daily_neg.text.strip() if daily_neg else np.nan)
        
        pagenum.append(i)  # Record the page number for this row
        
    # Print page completion time
    print(f'Page {i} completed in {time.time()-start_time:.2f} seconds')

print("Total Time Completed in seconds", str(time.time()-total_time))

In [ ]:
groww_df = pd.DataFrame({"company":companylist,"marketprice":marketpricelist,"closeprice":closepricelist,"marketcap":marketcaplist,"daily":dailylist,"daily_neglist":daily_neglist})

In [ ]:
groww_df

In [ ]:
groww_df.to_csv("groww2.csv")

In [ ]:
df = pd.read_csv("groww2.csv")

In [ ]:
# So what happened is, after getting the data using pagination scraping,
# we felt that the data was limited. If we could get more data,
# we would be able to perform more analysis.

# There is a search page on the website.
# After clicking it and entering a company name,
# we can see much more detailed information.

# There we observed that the URL contains slugs.
# Example: Reliance Industries -> reliance-industries-ltd

# But to do this for all companies,
# we initially thought we would need an API.

# However, after checking "View Page Source",
# we saw "__NEXT_DATA__", which means they are using Next.js (React framework).

# That means ALL stock data is embedded in the page as JSON.

# Groww uses Next.js, which often embeds page data inside:

# <script id="__NEXT_DATA__" type="application/json">

# So the frontend does not need to make an extra API call.

# This means Groww is embedding the entire stock data inside the page as JSON.

# This is MUCH better than using a hidden API.
# No authentication issues.
# Much safer.
# Perfect for academic scraping.

# We tested this with one company — Reliance Industries.

# Since the data is embedded,
# we used the company names obtained from pagination scraping
# and created slugs using the 're' library.

In [4]:
import requests
from bs4 import BeautifulSoup
import json # Library to work with JSON data

# testing on a specific stock page
# each company on Groww has a unique slug (like reliance-industries-ltd)
url = "https://groww.in/stocks/reliance-industries-ltd"

# set headers for request
# some websites block requests that don’t look like real browsers.
# Adding a User-Agent makes the request appear as if it’s coming from a browser, preventing blocks.
headers = {"User-Agent": "Mozilla/5.0"}


response = requests.get(url, headers=headers) # GET request to the company page
soup = BeautifulSoup(response.text, "html.parser") # parse HTML response
# response.text contains the full HTML content of the page.
# BeautifulSoup parses the HTML so you can extract elements easily.


# Find the __NEXT_DATA__ script tag
script_tag = soup.find("script", {"id": "__NEXT_DATA__"})
# Groww uses Next.js, which embeds all necessary page data inside a <script> tag with id="__NEXT_DATA__".
# Instead of scraping individual HTML elements, you can directly extract structured JSON


data = json.loads(script_tag.string)
# whats happening 
# Imagine the webpage hid all the company info inside a big box of LEGO pieces.
# But the box is locked in a special format called JSON — it’s like a secret code telling us how the pieces are organized.
# script_tag.string is like the note inside the box, written in that secret code.
# json.loads() is a magic key that opens the box and turns the note into a Python dictionary — a type of map where you can say “give me the company name” or “give me the price” easily.

stock_data = data["props"]["pageProps"]["stockData"]
# The box has layers of smaller boxes inside it.
# data["props"] → opens the first small box.
# ["pageProps"] → opens the next small box.
# ["stockData"] → finally gets the box with the actual stock information like company details, prices, fundamentals, etc.

print(stock_data.keys())
# This is like looking at all the labels on the LEGO pieces inside the box.
# It shows what kind of information you can get, like:
# fundamentals → money stuff, profits, revenue, etc.
# priceData → current price, yesterday’s price, etc.
# details → basic info about the company.



# simple analogy
# The webpage hides info in a locked treasure chest (JSON).
# json.loads() → opens the chest.
# ["props"]["pageProps"]["stockData"] → goes inside the chest to find the treasure (stock info).
# .keys() → looks at all the types of treasures you have.

dict_keys(['header', 'details', 'brandDtos', 'stats', 'fundamentals', 'shareHoldingPattern', 'fundsInvested', 'priceData', 'financialStatement', 'financialStatementV2', 'similarAssets'])


In [ ]:
# then from all these we picked, fundamentals, priceData and details
# fundamentals - the companies report card
# contains things like revenue, profit, EPS (earnings per share), P/E (price/earnings) ratio
# this tells you how healthy the company is financially.
# Essential for investors or anyone analyzing stocks.

# priceData - "the company’s price info"
# Contains current price, previous close, daily changes.
# You need this to know how the stock is moving in the market.

# details → "the company's basic info"
# Contains things like industry, sector, company name.
# Helps you identify the stock and group it correctly.

In [9]:
stock_data["fundamentals"]

[{'name': 'Market Cap', 'shortName': 'Mkt Cap', 'value': '₹19,21,682Cr'},
 {'name': 'ROE', 'shortName': 'ROE', 'value': '9.47%'},
 {'name': 'P/E Ratio(TTM)', 'shortName': 'P/E Ratio(TTM)', 'value': '19.65'},
 {'name': 'EPS(TTM)', 'shortName': 'EPS(TTM)', 'value': '72.25'},
 {'name': 'P/B Ratio', 'shortName': 'P/B Ratio', 'value': '2.19'},
 {'name': 'Dividend Yield', 'shortName': 'Div Yield', 'value': '0.39%'},
 {'name': 'Industry P/E', 'shortName': 'Industry P/E', 'value': '15.77'},
 {'name': 'Book Value', 'shortName': 'Book Value', 'value': '648.28'},
 {'name': 'Debt to Equity', 'shortName': 'Debt to Equity', 'value': '0.43'},
 {'name': 'Face Value', 'shortName': 'Face Value', 'value': '10'}]

In [6]:
stock_data["priceData"]

{'nse': {'yearLowPrice': 1114.85,
  'yearHighPrice': 1611.8,
  'close': 1419.4,
  'dayChange': 8.599999999999909,
  'dayChangePerc': 0.6058898125968655,
  'high': 1434.9,
  'highPriceRange': 1561.3,
  'highTradeRange': None,
  'lastTradeQty': 1,
  'lastTradeTime': 1771842527,
  'low': 1418.3,
  'lowPriceRange': 1277.5,
  'lowTradeRange': None,
  'ltp': 1428,
  'oiDayChange': 0,
  'oiDayChangePerc': 0,
  'open': 1425,
  'openInterest': None,
  'prevOpenInterest': None,
  'symbol': 'RELIANCE',
  'totalBuyQty': 0,
  'totalSellQty': 5152,
  'tsInMillis': 1771842539,
  'volume': 7758856,
  'type': 'LIVE_PRICE'},
 'bse': {'yearLowPrice': 1115.55, 'yearHighPrice': 1611.2}}

In [10]:
stock_data["header"]

{'searchId': 'reliance-industries-ltd',
 'growwCompanyId': 'GSTK500325',
 'isin': 'INE002A01018',
 'industryId': 52,
 'industryName': 'Refineries',
 'displayName': 'Reliance Industries',
 'shortName': 'Reliance Industries',
 'type': 'STOCK',
 'isFnoEnabled': True,
 'nseScriptCode': 'RELIANCE',
 'bseScriptCode': '500325',
 'nseTradingSymbol': 'RELIANCE-EQ',
 'bseTradingSymbol': 'RELIANCE',
 'isBseTradable': True,
 'isNseTradable': True,
 'logoUrl': 'https://assets-netstorage.groww.in/stock-assets/logos2/RELIANCE.webp',
 'floatingShares': 3405239713,
 'isBseFnoEnabled': False,
 'isNseFnoEnabled': True}

In [2]:
import re # Import the regular expressions library to clean and manipulate strings
def generate_slug(company_name):
    # convert the company name to lowercase, bacause URLS are usually lower case
    name = company_name.lower()
    
    # replace "&" with a space to avoid issues in the URL
    name = name.replace("&", " ")

    # Remove any character that is not a lowercase letter, number, or space
    # This cleans out punctuation like commas, periods, parentheses, etc.
    name = re.sub(r'[^a-z0-9 ]', '', name)

    # Replace multiple spaces with a single space and remove leading/trailing spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    #  Replace spaces with hyphens (to match Groww URL structure)
    # Append '-ltd' at the end because Groww uses this convention for company URLs
    return name.replace(" ", "-") + "-ltd"

In [8]:
df_new = pd.read_csv(r"C:\Users\91891\Downloads\part1.csv")

In [3]:
df_new = pd.read_csv("groww2.csv")

In [9]:
import pandas as pd
import requests
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


test_df = df_new.copy() # creates a copy of the df, prevents modifying original dataset.

# apply the generate_slug() function to each company name, creates a new col slug
# test_df["slug"] = test_df["company"].apply(generate_slug) 

# construct full company urls using the slug
# now each row has its own detailed stock page URL.
test_df["url"] = "https://groww.in/stocks/" + test_df["slug"]

# creating a Session (performace optimization)
# Instead of calling requests.get() repeatedly:
# Session() reuses the same TCP connection.
# Faster.
# More efficient.
# Looks more natural to the server.

session = requests.Session()


# headers --> mimics a real browser, 
# but why we need this
# prevents blocking
# keep-alive helps reuse connections
# makes scraping safer.
HEADERS = {
    # Mimics a real browser to prevent blocking
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9", # sets preferred language for response
    "Connection": "keep-alive" # keeps connection for reuse.
}


# function to check a single URL
def check_url(url, retries=2):
    # Checks if a given URL is accessible
    # retries parameter allows retrying failed requests
    for attempt in range(retries):
        try:
            # Add small random delay to mimic human behavior
            time.sleep(random.uniform(0.2, 0.6))

            # Send GET request with headers and timeout
            response = session.get(url, headers=HEADERS, timeout=6)

            # If request successful, return status code
            if response.status_code == 200:
                return 200
            else:
                return response.status_code

        except requests.exceptions.RequestException:
            # Handles connection errors, timeouts, etc.
            
            if attempt == retries - 1:
                # If final retry fails, return None
                return None
            time.sleep(1)  # Wait 1 second before retrying


# function to check multiple URLs in parallel

# Checks a list of URLs using multithreading
# max_workers = number of parallel threads
# what is a thread?
# normally python does task 1, then task 2 then task 3 one by one
# but thread allows task1,2,3 all running at the same time
# batch_size = number of URLs processed before pause


def check_urls_parallel(urls, max_workers=8, batch_size=200):

    # Pre-create results list to maintain correct index positions
    # example we have urls as [u1,u2,u3,u4] we are creating [None,None,None,None]
    # creating an empty list has the same length as urls
    # keeps order
    # will store status codes for later
    results = [None] * len(urls)

    # this is the batching logic
    # range(start, stopm step)
    # start at 0, go until len(urls) and jump in steps of batch_size
    for i in range(0, len(urls), batch_size):
        # Process URLs in batches to avoid overloading the server

        # Extract a slice of URLs for current batch
        batch = urls[i:i + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # only 8 urls will be checked at the same time,
            # even if bbatch has 200 urls, only 8 run simultaneously
            # others wait in queue
            # Create thread pool with defined number of workers
            future_to_index = {
                executor.submit(check_url, url): i + idx
                for idx, url in enumerate(batch) # here we get {0 :"u1", 1:"u2"}
                # but 0, 1 are not original so we use + i, so we get original index
            }
            # thread donot finish in order, url 5 finishes first, url 2 finishes second
            # so as_completed gives them in finishing order.
            # Submit URL checking tasks to thread pool
            # Map each future object to its original index

            for future in as_completed(future_to_index):
                # As each thread completes
                index = future_to_index[future]
                # Get original index
                results[index] = future.result()
                # Store returned status code in correct position
                
        # Pause between batches to prevent server overload (very important)
        time.sleep(3)

    return results
    # Return list of status codes

    
# Convert URL column to list and check all URLs
# Store HTTP status codes in new column "status"
test_df["status"] = check_urls_parallel(test_df["url"].tolist())

print(test_df[["company", "url", "status"]])
# Display company name, generated URL, and its status code.

            company                                                url  status
0               SBI        https://groww.in/stocks/state-bank-of-india     200
1               TCS  https://groww.in/stocks/tata-consultancy-servi...     200
2               L&T          https://groww.in/stocks/larsen-toubro-ltd     200
3               LIC  https://groww.in/stocks/life-insurance-corpora...     200
4     Maruti Suzuki    https://groww.in/stocks/maruti-suzuki-india-ltd     200
...             ...                                                ...     ...
1817        Elegant  https://groww.in/stocks/elegant-marbles-and-gr...     404
1818         Hybrid  https://groww.in/stocks/hybrid-finance-service...     404
1819          Aeonx  https://groww.in/stocks/aeonx-digital-technolo...     404
1820        Bhaskar      https://groww.in/stocks/bhaskar-agro-chem-ltd     404
1821         Vineet    https://groww.in/stocks/vineet-laboratories-ltd     200

[1822 rows x 3 columns]


In [ ]:
# so why we used sessions
# imagine requests.get(url)
# what is does
# opens a connection to the server
# sends the requests
# gets the repsonse
# close the connection
# now imagine doing that 1000 times
# session = requests.Session()
# its like saying to the server, i will talk to u many times, let's keep the connection open
# so it reuses the same TCP connection
# saves time
# reduces overhead
# looks more like a real browser
# what is TCP connection? Transfer control protocol
# When your computer talks to a website:
# It first establishes a connection (like a handshake).
# That handshake takes time.
# Session keeps that handshake alive.
# That’s what "Connection": "keep-alive" also helps with.


# Why do we need headers?
# Because websites check:
# "Is this a real browser?"
# If you don't send headers:
# You look like a bot.
# User-Agent tells the server:
# "I'm Chrome browser."
# Accept-Language says:
# "I prefer English."
# Connection keep-alive says:
# "Keep connection open."

In [5]:
valid_df = test_df[test_df["status"] == 200].copy()
st_200 = valid_df.to_csv("st_200.csv")

In [10]:
import requests
import json
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive"
}

def extract_selected_data(url, retries=2):
    for attempt in range(retries):
        try:
            time.sleep(random.uniform(0.3, 0.8))  # small random delay

            response = session.get(url, headers=HEADERS, timeout=8)

            match = re.search(
                r'<script id="__NEXT_DATA__" type="application/json".*?>(.*?)</script>',
                response.text
            )

            if not match:
                return None

            data = json.loads(match.group(1))
            stock_data = data["props"]["pageProps"]["stockData"]

            return {
                "fundamentals": stock_data.get("fundamentals", {}),
                "priceData": stock_data.get("priceData", {}),
                "details": {
                    "header": stock_data.get("header", {}),
                    "about": stock_data.get("about", {})
                }
            }

        except Exception:
            if attempt == retries - 1:
                return None
            time.sleep(1)

def extract_parallel(urls, max_workers=6, batch_size=150):
    results = [None] * len(urls)

    for i in range(0, len(urls), batch_size):
        batch = urls[i:i + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_index = {
                executor.submit(extract_selected_data, url): i + idx
                for idx, url in enumerate(batch)
            }

            for future in as_completed(future_to_index):
                index = future_to_index[future]
                results[index] = future.result()

        print(f"Finished batch {i} to {i+batch_size}")
        time.sleep(5)  # batch pause

    return results

# Apply only on valid URLs
valid_urls = test_df[test_df["status"] == 200]["url"].tolist()
extracted_data = extract_parallel(valid_urls)

Finished batch 0 to 150
Finished batch 150 to 300
Finished batch 300 to 450
Finished batch 450 to 600
Finished batch 600 to 750
Finished batch 750 to 900
Finished batch 900 to 1050
Finished batch 1050 to 1200
Finished batch 1200 to 1350


In [11]:
import pandas as pd

# List of keys we want from fundamentals
fund_keys = {
    "Mkt Cap": "fund_Market_Cap",
    "ROE": "fund_ROE",
    "P/E Ratio(TTM)": "fund_P_E_RatioTTM",
    "EPS(TTM)": "fund_EPSTTM",
    "P/B Ratio": "fund_P_B_Ratio",
    "Div Yield": "fund_Dividend_Yield",
    "Industry P/E": "fund_Industry_P_E",
    "Book Value": "fund_Book_Value",
    "Debt to Equity": "fund_Debt_to_Equity"
}


final_data = []

for item in extracted_data:
    if item is None:
        continue

    row = {}

    # Company Name
    row["company_name"] = item.get("details", {}).get("header", {}).get("displayName")

    # Extract fundamentals
    fundamentals = item.get("fundamentals", [])
    for f in fundamentals:
        key = f.get("shortName")
        if key in fund_keys:
            row[fund_keys[key]] = f.get("value")

    # Extract priceData
    nse = item.get("priceData", {}).get("nse", {})
    bse = item.get("priceData", {}).get("bse", {})

    # instead od row["price_nse_ltp"] = nse.get("ltp") instead of writing multiple time
    # row[something] use row.update
    row.update({
        "price_nse_ltp": nse.get("ltp"),
        "price_nse_open": nse.get("open"),
        "price_nse_close": nse.get("close"),
        "price_nse_high": nse.get("high"),
        "price_nse_low": nse.get("low"),
        "price_nse_volume": nse.get("volume"),
        "price_nse_dayChange": nse.get("dayChange"),
        "price_nse_dayChangePerc": nse.get("dayChangePerc"),
        "price_nse_yearLowPrice": nse.get("yearLowPrice"),
        "price_nse_yearHighPrice": nse.get("yearHighPrice"),
        "price_bse_yearLowPrice": bse.get("yearLowPrice"),
        "price_bse_yearHighPrice": bse.get("yearHighPrice")
    })

    # Extract details
    header = item.get("details", {}).get("header", {})
    about = item.get("details", {}).get("about", {})

    row.update({
        "details_industryName": header.get("industryName"),
        "details_isFnoEnabled": header.get("isFnoEnabled"),
        "details_floatingShares": header.get("floatingShares"),
        "details_isNseFnoEnabled": header.get("isNseFnoEnabled")
    })

    final_data.append(row) # for every company

# Convert to DataFrame
final_df = pd.DataFrame(final_data)

# Save
final_df.to_csv("day_5_batch_2.csv", index=False)

print(final_df.head())

                               company_name fund_Market_Cap fund_ROE  \
0                       State Bank of India    ₹11,22,582Cr   14.34%   
1                 Tata Consultancy Services     ₹9,72,053Cr   46.46%   
2                           Larsen & Toubro     ₹6,02,552Cr   16.18%   
3  Life Insurance Corporation of India Ltd.     ₹5,52,204Cr    0.00%   
4                       Maruti Suzuki India     ₹4,71,212Cr   14.82%   

  fund_P_E_RatioTTM fund_EPSTTM fund_P_B_Ratio fund_Dividend_Yield  \
0             12.97       93.75           1.99               1.26%   
1             20.27      132.56           9.19               4.69%   
2             31.75      137.95           5.94               0.78%   
3             10.42       83.79           3.89               1.37%   
4             31.56      474.92           4.72               0.90%   

  fund_Industry_P_E fund_Book_Value fund_Debt_to_Equity  ...  \
0             14.57          611.73                  NA  ...   
1             21.6

In [ ]:
# Save your DataFrame to CSV
final_df.to_csv("groww_final_stock_data_2ndbatch.csv", index=False)

print("CSV saved successfully!")

In [ ]:
df = pd.read_csv("groww_final_stock_data.csv")